# Lenormand B16.1 — Confirmed Seven-Label Latent Test Ensemble

本 notebook 部署已经通过 Fold 1/2 untouched confirmation 的 B16.1：

- 三个冻结 Qwen3.8-27B Factor fold adapter；
- 三个固定 `Layer 63 / C=0.001` latent probe；
- 仅替换七个预注册 Factor；
- 当前线上最佳 Subtask 1 和另外17个 Factor 逐单元格保持不变；
- 使用与当前线上 `0.6542` Factor 相同思想的 fold-normalized margin ensemble。

这里**不重新训练一个未经验证的 full-data adapter**。只需抽取 `378 × 7 × 3 = 7,938` 个 hidden vectors。A100 80GB 预计35–60分钟，每128个prompt写入Drive，可断线续跑。


In [ ]:
#@title 0A. 新runtime安装依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.6.0,<1.8.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. Qwen3.8 kernels（之后重启session）
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation
print('Runtime → Restart session；重启后从第1格开始。')


In [ ]:
#@title 1. Drive、冻结产物和续跑开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import gc, importlib, json, shutil, subprocess, sys, time

ROOT=Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH=ROOT/'train.xlsx'
TEST_PATH=ROOT/'leaderboard.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH=ROOT/'ieee/train.xlsx'
if not TEST_PATH.exists(): TEST_PATH=ROOT/'ieee/leaderboard.xlsx'
Q14_OOF=ROOT/'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'
Q38_OOF=ROOT/'results/B4_Q38F_FULL64_THREE_FOLD_OOF/Q38_FULL64_OOF.npz'
FULL64_CONFIG=ROOT/'results/B4_Q38F_FULL64_THREE_FOLD_OOF/FULL64_CONFIG.json'
FACTOR_ROOT=ROOT/'results/B4_Q38F_FULL64_KERNEL_FOLD0/FULL64_FACTOR_FOLD0'
B16_ROOT=ROOT/'results/B16_FACTOR_LATENT_GATE/fold_0'
B161_ROOT=ROOT/'results/B161_FACTOR_LATENT_ROUTE_CONFIRMATION'
FINAL_ROOT=ROOT/'results/B4_FINAL_SUBMISSION'
Q38_TEST_ROOT=FINAL_ROOT/'FACTOR_TEST/Q38'
OUT=ROOT/'results/B161_FACTOR_LATENT_TEST_ENSEMBLE'
OUT.mkdir(parents=True,exist_ok=True)

# 当前线上最佳：Subtask1=.8031, Subtask2=.6542。如果Drive没有就会提示上传。
SOURCE_SUBMISSION=ROOT/'results/B151_SAFE_FACTOR6542_HYBRID/Lenormand.csv'
if not SOURCE_SUBMISSION.exists():
    print('请上传当前线上最佳 Lenormand.csv（0.8031 / 0.6542）')
    uploaded=files.upload()
    if 'Lenormand.csv' not in uploaded: raise FileNotFoundError('Lenormand.csv')
    SOURCE_SUBMISSION=OUT/'SOURCE_PUBLIC_BEST.csv'
    shutil.copy2('/content/Lenormand.csv',SOURCE_SUBMISSION)

# 可按 (0,) / (1,) / (2,) 分session跑；已经完成的fold自动resume。
FOLDS_TO_RUN=(0,1,2)

MODULE_MARKERS={
 'b1_experiments.py':None,
 'b4p_anchor_verifier.py':'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
 'b7_top8_sprint.py':'B7_RUNTIME_REVISION = "2026-08-27.top8-sprint-v1"',
 'b15_latent_readout.py':'B15_RUNTIME_REVISION = "2026-08-30.latent-risk-readout-v1"',
 'b16_factor_latent_readout.py':'B16_RUNTIME_REVISION = "2026-08-30.factor-hidden-readout-gate-v1"',
 'b161_factor_latent_route.py':'B161_RUNTIME_REVISION = "2026-08-31.factor-latent-seven-label-confirmation-v3"',
 'b161_factor_latent_test.py':'B161_TEST_RUNTIME_REVISION = "2026-08-31.factor-latent-test-ensemble-v1"',
}
stale=[]
for name,marker in MODULE_MARKERS.items():
    path=ROOT/name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('上传并覆盖：',stale)
    uploaded=files.upload()
    for name in stale:
        if name not in uploaded: raise FileNotFoundError(name)
        shutil.copy2('/content/'+name,ROOT/name)

required=[TRAIN_PATH,TEST_PATH,Q14_OOF,Q38_OOF,FULL64_CONFIG,SOURCE_SUBMISSION,
          B16_ROOT/'B16_FOLD_OUTPUTS.npz',B16_ROOT/'B16_PROBE.joblib',
          B161_ROOT/'B161_CONFIRMATION_DECISION.json']
for fold in range(3):
    required += [FACTOR_ROOT/f'fold_{fold}/verifier/adapter_final/adapter_config.json',
                 Q38_TEST_ROOT/f'fold_{fold}/factor_test_logits.npz']
for fold in (1,2):
    required += [B161_ROOT/f'fold_{fold}/B161_PROBE.joblib',
                 B161_ROOT/f'fold_{fold}/B161_FOLD_OUTPUTS.npz']
missing=[str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError('缺少冻结产物：\n'+'\n'.join(missing))
sys.path.insert(0,str(ROOT))
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
print({'scheduled':FOLDS_TO_RUN,'source':str(SOURCE_SUBMISSION),'out':str(OUT),
       'free_gb':round(shutil.disk_usage(ROOT).free/2**30,2)})


In [ ]:
#@title 2. 环境、数据与B16.1 PASS锁定
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

import b1_experiments as b1
import b4p_anchor_verifier as b4
import b7_top8_sprint as b7
import b15_latent_readout as b15
import b16_factor_latent_readout as b16
import b161_factor_latent_route as b161
import b161_factor_latent_test as b161t
for module in (b1,b4,b7,b15,b16,b161,b161t): importlib.reload(module)
assert b161t.B161_TEST_RUNTIME_REVISION=='2026-08-31.factor-latent-test-ensemble-v1'

gpu_gb=torch.cuda.get_device_properties(0).total_memory/2**30
kernel=b4.qwen35_kernel_status()
print({'transformers':transformers.__version__,'torch':torch.__version__,
       'sklearn':sklearn.__version__,'gpu_gb':gpu_gb,'kernel':kernel,
       'b161_test':b161t.B161_TEST_RUNTIME_REVISION})
assert gpu_gb>=70
assert kernel['causal_conv1d'] and kernel['flash_linear_attention'], '运行0B并重启session'
torch.set_float32_matmul_precision('high')

decision=json.loads((B161_ROOT/'B161_CONFIRMATION_DECISION.json').read_text(encoding='utf-8'))
assert decision['passed'] and decision['decision']=='BUILD_B161_TEST_ROUTE'
assert decision['selected_layer']==63 and abs(decision['c_value']-0.001)<1e-12
assert tuple(decision['route_labels'])==b161.ROUTE_LABELS

bundle=b1.load_training_data(ROOT,TRAIN_PATH)
anchor=b16.load_factor_anchor(bundle,Q14_OOF,Q38_OOF,q38_weight=0.75)
folds=anchor['folds']
test_corpus=b4.load_test_data(ROOT,TEST_PATH)
test_frame=test_corpus.frame.copy().fillna('')
source=pd.read_csv(SOURCE_SUBMISSION,dtype=str,keep_default_na=False)
source_audit=b7.audit_submission(source,test_frame)
verifier_cfg=b16.load_full64_config(FULL64_CONFIG)
route_ids=b161._route_ids(b161.RouteConfirmationConfig())
print({'train_rows':len(bundle.texts),'test_rows':len(test_corpus.texts),
       'fold_sizes':np.bincount(folds).tolist(),'route_ids':route_ids.tolist(),
       'route_labels':[b1.FACTOR_LABELS[i] for i in route_ids],
       'source_audit':source_audit})


In [ ]:
#@title 3. 复用Q38 train/test语义缓存
train_corpus=b4.training_corpus(bundle)
train_cache=b4.prepare_semantic_cache(
    train_corpus,verifier_cfg,FINAL_ROOT/'SEMANTIC_CACHE/Q38_TRAIN')
test_cache=b4.prepare_semantic_cache(
    test_corpus,verifier_cfg,FINAL_ROOT/'SEMANTIC_CACHE/Q38_TEST')
print('Semantic caches ready.')


In [ ]:
#@title 4. 重建三折冻结threshold与严格OOF审计（CPU）
state=b161t.load_route_deployment_state(
    bundle=bundle,
    anchor=anchor,
    b16_fold0_output=B16_ROOT/'B16_FOLD_OUTPUTS.npz',
    b161_root=B161_ROOT,
    route_ids=route_ids,
)
print({'strict_oof_route_metrics':state['metrics'],
       'thresholds':{fold:state['thresholds'][fold][route_ids].tolist() for fold in range(3)}})


In [ ]:
#@title 5. 三折test latent inference（断线后重跑本格即可）
PROBES={
 0:B16_ROOT/'B16_PROBE.joblib',
 1:B161_ROOT/'fold_1/B161_PROBE.joblib',
 2:B161_ROOT/'fold_2/B161_PROBE.joblib',
}
fold_probabilities={}
for fold in range(3):
    cached=OUT/f'fold_{fold}/B161_TEST_FOLD_OUTPUTS.npz'
    if fold not in FOLDS_TO_RUN and not cached.exists():
        print(f'[Fold {fold}] 未安排且无缓存。')
        continue
    started=time.perf_counter()
    result=b161t.score_test_fold(
        bundle=bundle,
        folds=folds,
        test_corpus=test_corpus,
        train_cache=train_cache,
        test_cache=test_cache,
        verifier_config=verifier_cfg,
        adapter_path=FACTOR_ROOT/f'fold_{fold}/verifier/adapter_final',
        probe_path=PROBES[fold],
        q38_test_logits_path=Q38_TEST_ROOT/f'fold_{fold}/factor_test_logits.npz',
        output_dir=OUT/f'fold_{fold}',
        fold=fold,
        route_ids=route_ids,
        selected_layer=63,
        extraction_batch_size=2,
        extraction_chunk_size=128,
    )
    fold_probabilities[fold]=result['probability']
    print({'fold':fold,'minutes':round((time.perf_counter()-started)/60,1),
           'q38_reproduction_mae':result['q38_reproduction_mae']})
    gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 6. 三折margin ensemble并生成唯一推荐CSV
for fold in range(3):
    if fold not in fold_probabilities:
        saved=np.load(OUT/f'fold_{fold}/B161_TEST_FOLD_OUTPUTS.npz',allow_pickle=True)
        assert saved['row_ids'].astype(str).tolist()==test_corpus.row_ids.astype(str).tolist()
        fold_probabilities[fold]=saved['route_probability'].astype(np.float32)

ensemble=b161t.ensemble_route(
    fold_probabilities=fold_probabilities,
    thresholds=state['thresholds'],
    route_ids=route_ids,
)
np.savez_compressed(
    OUT/'B161_TEST_ENSEMBLE_OUTPUTS.npz',
    row_ids=test_corpus.row_ids.astype(str),
    route_ids=route_ids.astype(np.int16),
    **ensemble,
)
audit=b161t.build_submission(
    source_submission=SOURCE_SUBMISSION,
    test_frame=test_frame,
    route_prediction=ensemble['prediction'],
    route_ids=route_ids,
    output_dir=OUT/'01_B161_THREEFOLD_LATENT_ROUTE',
)
audit.update({
    'validated_oof_delta_macro_f1':decision['deltas']['macro_f1'],
    'validated_oof_delta_macro_ap':decision['deltas']['macro_ap'],
    'validated_oof_delta_tail_macro_f1':decision['deltas']['tail_macro_f1'],
    'public_source_subtask1':0.8031,
    'public_source_subtask2':0.6542,
    'projected_subtask2':0.6542+decision['deltas']['macro_f1'],
    'projected_composite':0.7*0.8031+0.3*(0.6542+decision['deltas']['macro_f1']),
})
b161t.json_dump(audit,OUT/'01_B161_THREEFOLD_LATENT_ROUTE/AUDIT.json')
display(pd.DataFrame([audit]))
print('CSV:',audit['path'])
print('这是唯一推荐上传的B16.1候选；Task1与17个非route Factors保持不变。')


In [ ]:
#@title 7. 最终审计、打包并下载
candidate=OUT/'01_B161_THREEFOLD_LATENT_ROUTE/Lenormand.csv'
reread=pd.read_csv(candidate,dtype=str,keep_default_na=False)
final_audit=b7.audit_submission(reread,test_frame,source)
assert final_audit['changed_risk_level_rows']==0
assert final_audit['changed_evidence_rows']==0
assert final_audit['verbatim_failures']==0
assert final_audit['indicator_nonempty_evidence']==0
print(final_audit)

package=Path('/content/B161_TEST_SUBMISSION_PACKAGE')
if package.exists(): shutil.rmtree(package)
package.mkdir(parents=True)
shutil.copytree(OUT/'01_B161_THREEFOLD_LATENT_ROUTE',package/'01_B161_THREEFOLD_LATENT_ROUTE')
for name in ('B161_TEST_ENSEMBLE_OUTPUTS.npz',): shutil.copy2(OUT/name,package/name)
shutil.copy2(B161_ROOT/'B161_CONFIRMATION_DECISION.json',package/'B161_CONFIRMATION_DECISION.json')
archive=shutil.make_archive('/content/B161_TEST_SUBMISSION_PACKAGE','zip',package)
print('Package:',archive)
print('比赛上传文件：01_B161_THREEFOLD_LATENT_ROUTE/Lenormand.csv；不要上传zip。')
files.download(archive)
